In [6]:
file_path = r"C:\Users\Admin Lenovo\Downloads\Sample.xlsx"

TP-1: CleaningData — Applying OOP and Design Patterns

In [12]:
import pandas as pd
from io import StringIO


class CSVReader:

    def __init__(self, data):
        self.data = data
        self.df = None

    def read(self):
        self.df = pd.read_csv(StringIO(self.data), sep="\t")
        return self.df

    def preview(self, n=5):
        print(self.df.head(n))


# Sample data
data = """id	name	age	height_cm	weight_kg	city	score
1	Alice	29	165	68	New York	85
2	Bob	NaN	172	NaN	Los Angeles	90
3	Charlie	35	168	72	Chicago	NaN
4	David	NaN	NaN	80	Houston	75
5	Eva	27	160	55	New York	88
6	Frank	30	175	85	Los Angeles	NaN
7	Grace	NaN	162	60	Chicago	92"""

# Test Exercise 1
reader = CSVReader(data)

reader.read()
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


Exercise 2: Strategy Pattern for Missing Values

In [13]:
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):

    def handle(self, df):
        return df.dropna()


class FillMean(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        columns = df.select_dtypes(
            include="number"
        ).columns.drop("id", errors="ignore")

        df[columns] = df[columns].fillna(
            df[columns].mean()
        )

        return df


class FillMode(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        for column in df.columns:
            if df[column].isnull().any():

                mode_values = df[column].mode()

                if not mode_values.empty:
                    df[column] = df[column].fillna(
                        mode_values.iloc[0]
                    )

        return df


class DataCleaner:

    def __init__(self, strategy: MissingValueStrategy):
        self._strategy = strategy

    def set_strategy(self, strategy):
        self._strategy = strategy

    def clean(self, df):
        return self._strategy.handle(df)


# Test Exercise 2
cleaner = DataCleaner(FillMean())

cleaned_df = cleaner.clean(reader.df)

print(cleaned_df)

   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


In [14]:
cleaner = DataCleaner(DropMissing())

In [15]:
cleaner = DataCleaner(DropMissing())

In [16]:
cleaner = DataCleaner(FillMode())

Exercise 3: Decorators for Logging and Timing

In [17]:
import time
from functools import wraps


def log_action(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting: {func.__name__}")

        result = func(*args, **kwargs)

        print(f"Finished: {func.__name__}")

        return result

    return wrapper


def log_time(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()

        print(f"Execution time: {end - start:.4f} seconds")

        return result

    return wrapper

In [18]:
class CSVReaderWithLogging:

    def __init__(self, data):
        self.data = data
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(
            StringIO(self.data),
            sep="\t"
        )
        return self.df

    def preview(self, n=5):
        print(self.df.head(n))


# Test Exercise 3
reader3 = CSVReaderWithLogging(data)

reader3.read()
reader3.preview(5)

Starting: read
Execution time: 0.0036 seconds
Finished: read
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


Exercise 4: Factory Pattern for Data Transformations

In [19]:
class DataTransform(ABC):

    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):

    def apply(self, df):
        df = df.copy()

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        return df


class RemoveDuplicates(DataTransform):

    def apply(self, df):
        return df.drop_duplicates()


class StandardizeText(DataTransform):

    def apply(self, df):
        df = df.copy()

        text_columns = df.select_dtypes(
            include=["object", "string"]
        ).columns

        for column in text_columns:
            df[column] = df[column].apply(
                lambda x: x.strip().title()
                if isinstance(x, str)
                else x
            )

        return df


class TransformFactory:

    @staticmethod
    def get_transform(name):

        transforms = {
            "normalize": NormalizeColumns,
            "duplicates": RemoveDuplicates,
            "text": StandardizeText
        }

        if name not in transforms:
            raise ValueError(
                f"Unknown transformation: {name}"
            )

        return transforms[name]()


# Test Exercise 4
transform = TransformFactory.get_transform("normalize")

result = transform.apply(reader.df)

print(result.head())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


Exercise 5: Build the Full Cleaning Pipeline

In [20]:
class DataPipeline(ABC):

    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):

    def __init__(self, data, output_path):
        self.data = data
        self.output_path = output_path
        self.df = None

    def load(self):
        print("\n1. Loading data...")

        reader = CSVReader(self.data)

        self.df = reader.read()

        print("Data loaded successfully.")

    def clean(self):
        print("\n2. Cleaning missing values...")

        cleaner = DataCleaner(FillMean())

        self.df = cleaner.clean(self.df)

        print("Missing values handled.")

    def transform(self):
        print("\n3. Transforming data...")

        for name in ["normalize", "duplicates", "text"]:

            transform = TransformFactory.get_transform(name)

            self.df = transform.apply(self.df)

        print("Transformations completed.")

    def save(self):
        print("\n4. Saving data...")

        self.df.to_csv(
            self.output_path,
            index=False
        )

        print("Cleaning completed!")
        print(f"Saved file: {self.output_path}")


# Test Exercise 5
output_path = "cleaned_data.csv"

pipeline = CSVDataPipeline(data, output_path)

pipeline.run()

print("\nPreview of cleaned data:")
print(pipeline.df)


1. Loading data...
Data loaded successfully.

2. Cleaning missing values...
Missing values handled.

3. Transforming data...
Transformations completed.

4. Saving data...
Cleaning completed!
Saved file: cleaned_data.csv

Preview of cleaned data:
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0
